In [1]:
import subprocess
import time
import requests
from IPython.display import display, HTML

# Запускаем сервер через uvicorn
process = subprocess.Popen(
    ["uvicorn", "api:app", "--host", "127.0.0.1", "--port", "8000", "--log-level", "critical"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Ждём запуска (3 секунды)
time.sleep(3)

# Проверяем работоспособность
try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5)
    if response.status_code == 200:
        display(HTML('''
            <div style="background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 4px; padding: 15px; margin: 10px 0;">
                <h3 style="margin-top: 0; color: #155724;">✅ API сервер запущен</h3>
                <p><strong>Базовый URL:</strong> <a href="http://127.0.0.1:8000" target="_blank">http://127.0.0.1:8000</a></p>
                <p><strong>Документация (Swagger):</strong> <a href="http://127.0.0.1:8000/docs" target="_blank">http://127.0.0.1:8000/docs</a></p>
                <p><strong>Health check:</strong> <a href="http://127.0.0.1:8000/health" target="_blank">http://127.0.0.1:8000/health</a></p>
            </div>
        '''))

        # Сохраняем процесс в глобальную переменную для остановки
        globals()['api_process'] = process
    else:
        print(f"⚠️ Сервер ответил статусом {response.status_code}")
        process.terminate()
except Exception as e:
    print(f"❌ Ошибка запуска сервера: {e}")
    print("Возможные причины:")
    print("  1. Не установлен uvicorn: выполните !pip install uvicorn")
    print("  2. Не запущен модуль В.1 (нет файла ev_consumption_regressor.pkl)")
    print("  3. Порт 8000 занят другим приложением")
    try:
        process.terminate()
    except:
        pass

In [29]:
# Остановка сервера
try:
    process = globals().get('api_process')
    if process:
        process.terminate()
        process.wait(timeout=5)
        print("✅ API сервер остановлен")
    else:
        print("⚠️ Сервер не был запущен из этой сессии")
except Exception as e:
    print(f"Ошибка при остановке: {e}")

✅ API сервер остановлен


In [4]:
import requests

# Пример маршрута: 3 точки в Москве
coordinates = [
    {"latitude": 55.7558, "longitude": 37.6173},
    {"latitude": 55.7600, "longitude": 37.6200},
    {"latitude": 55.7650, "longitude": 37.6250}
]

payload = {
    "vehicle_id": "EV001",
    "coordinates": coordinates,
    "start_soc": 80.0,
    "avg_temperature_c": 20.0,
    "road_type": "highway",
    "avg_speed_kmh": 45.0
}

print("📍 ТЕСТ ЭНДПОИНТА /predict_consumption")
print("=" * 60)
print(f"Отправка запроса на маршрут из {len(coordinates)} точек...")
print(f"  • Начальный заряд: {payload['start_soc']}%")
print(f"  • Температура: {payload['avg_temperature_c']}°C")
print(f"  • Тип дороги: {payload['road_type']}")
print(f"  • Средняя скорость: {payload['avg_speed_kmh']} км/ч")
print("-" * 60)

try:
    response = requests.post(
        "http://127.0.0.1:8000/predict_consumption",
        json=payload,
        timeout=10
    )

    if response.status_code == 200:
        data = response.json()
        print("✅ ПРОГНОЗ УСПЕШНО РАССЧИТАН")
        print("=" * 60)
        print(f"Дистанция маршрута:  {data['route_distance_km']} км")
        print(f"Прогноз расхода:     {data['predicted_consumption_kwh_per_100km']} кВт⋅ч/100км")
        print(f"Энергия на маршрут:  {data['energy_needed_kwh']} кВт⋅ч")
        print(f"Конечный заряд:      {data['estimated_end_soc_percent']}%")
        print(f"Уровень риска:       {data['risk_level'].upper()}")
        print(f"Обработано точек:    {data['coordinates_processed']}")
        print(f"Модель:              {data['model_used']}")
        print("=" * 60)

        # Визуализация маршрута
        print("\n🗺️  КООРДИНАТЫ МАРШРУТА:")
        for i, coord in enumerate(coordinates, 1):
            print(f"  {i}. {coord['latitude']:.4f}, {coord['longitude']:.4f}")

    else:
        print(f"❌ ОШИБКА {response.status_code}")
        print(f"Ответ сервера: {response.text}")

except requests.exceptions.ConnectionError:
    print("❌ СЕРВЕР НЕ ЗАПУЩЕН")
    print("Выполните ячейку с запуском API перед тестированием")

except KeyError as e:
    print(f"❌ ОШИБКА КЛЮЧА: {e}")
    print("Поля в ответе сервера:")
    print(response.json().keys())

except Exception as e:
    print(f"❌ НЕИЗВЕСТНАЯ ОШИБКА: {e}")
    if 'response' in locals():
        print(f"Ответ сервера: {response.json()}")

📍 ТЕСТ ЭНДПОИНТА /predict_consumption
Отправка запроса на маршрут из 3 точек...
  • Начальный заряд: 80.0%
  • Температура: 20.0°C
  • Тип дороги: highway
  • Средняя скорость: 45.0 км/ч
------------------------------------------------------------
✅ ПРОГНОЗ УСПЕШНО РАССЧИТАН
Дистанция маршрута:  1.13 км
Прогноз расхода:     39.6 кВт⋅ч/100км
Энергия на маршрут:  0.45 кВт⋅ч
Конечный заряд:      79.4%
Уровень риска:       LOW
Обработано точек:    3
Модель:              SGDRegressor (huber)

🗺️  КООРДИНАТЫ МАРШРУТА:
  1. 55.7558, 37.6173
  2. 55.7600, 37.6200
  3. 55.7650, 37.6250


In [4]:
import requests

# Замените "EV001" на реальный ID из вашей БД (например, "vehicle_1")
vehicle_id = "EV003"

response = requests.get(f"http://127.0.0.1:8000/vehicle_risk/{vehicle_id}")

if response.status_code == 200:
    data = response.json()
    print(f"📊 ОЦЕНКА РИСКА ДЛЯ АВТОМОБИЛЯ {data['vehicle_id']}")
    print("=" * 50)
    print(f"Последний заряд:          {data['last_recorded_soc_percent']}%")
    print(f"Средний расход (10 поездок): {data['avg_consumption_last_10_trips']} кВт⋅ч/100км")
    print(f"Доля рискованных поездок: {data['high_consumption_ratio_percent']}%")
    print(f"Рекомендация:             {data['recommendation']}")
    print(f"Проанализировано поездок: {data['analyzed_trips']}")
    print("=" * 50)
else:
    print(f"❌ Ошибка {response.status_code}: {response.text}")
    print("\n💡 Совет: проверьте существующие ID автомобилей:")
    # Показать доступные ID из БД
    import sqlite3
    conn = sqlite3.connect('ev_championship.db')
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT vehicle_id FROM telematics_preprocessed LIMIT 5")
    print("Доступные vehicle_id:", [row[0] for row in cursor.fetchall()])
    conn.close()

📊 ОЦЕНКА РИСКА ДЛЯ АВТОМОБИЛЯ EV003
Последний заряд:          5.0%
Средний расход (10 поездок): 33.44 кВт⋅ч/100км
Доля рискованных поездок: 60.0%
Рекомендация:             🔴 КРИТИЧЕСКИ НИЗКИЙ ЗАРЯД! Немедленно подключите зарядку.
Проанализировано поездок: 10


In [5]:
import requests

vehicle_id = "EV002"

response = requests.get(f"http://127.0.0.1:8000/battery_health_forecast/{vehicle_id}")

if response.status_code == 200:
    data = response.json()
    print(f"🔋 ПРОГНОЗ БАТАРЕИ ДЛЯ {data['vehicle_id']}")
    print("=" * 50)
    print(f"Модель:               {data['model']}")
    print(f"Текущее здоровье:     {data['current_health_percent']}%")
    print(f"Пробег при поступлении: {data['odometer_at_entry_km']:,} км")
    print(f"Скорость деградации:  {data['degradation_rate_percent_per_1000km']}%/1000км")
    print(f"До порога 80%:        {data['remaining_km_to_80_percent']:,.0f} км")
    print(f"Прогноз через 1 год:  {data['forecast_health_1_year_percent']}%")
    print(f"\nРекомендация: {data['recommendation']}")
else:
    print(f"❌ Ошибка {response.status_code}: {response.text}")

🔋 ПРОГНОЗ БАТАРЕИ ДЛЯ EV002
Модель:               Mercedes EQC 400
Текущее здоровье:     71.0%
Пробег при поступлении: 21,398 км
Скорость деградации:  1.355%/1000км
До порога 80%:        0 км
Прогноз через 1 год:  50.7%

Рекомендация: 🔴 КРИТИЧЕСКИЙ ИЗНОС! Батарея ниже 80% — требуется замена.


In [5]:
# Запускаем сервер ПРЯМО В ЯЧЕЙКЕ (не через subprocess)
from flask import Flask, render_template_string
import threading

app = Flask(__name__)

@app.route('/')
def index():
    return "✅ Flask запущен! Порт 5000 работает."

def run_server():
    print("🚀 Запуск сервера на http://127.0.0.1:5000")
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

# Запускаем в фоновом потоке
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Ждём запуска
import time
time.sleep(2)

print("✅ Сервер запущен в фоновом потоке")
print("Откройте в браузере: http://127.0.0.1:5000")

🚀 Запуск сервера на http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


✅ Сервер запущен в фоновом потоке
Откройте в браузере: http://127.0.0.1:5000
